In [ ]:
from collections.abc import Callable, Iterable
import json
from pathlib import Path
import re

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData

from jazz_graph.data.graph_builder.graph_builder import CreateTensors
from jazz_graph.data.fetch import fetch_recording_traits, fetch_discogs_to_recording_id
from jazz_graph.model.model import LinkPredictionModel, UnsupervisedJazzModel
from jazz_graph.recommendation.recommender import Recommender, LookupRecordings, PredictLinkRecommender, RandomWalkRecommender
from jazz_graph.metrics.ranking import map_at_k
from jazz_graph.recommendation.experiment import BSideExperiment
from jazz_graph.training.logging import ExperimentLogger, load_embeddings, load_model, find_most_recent_run


In [ ]:
album_experiments = [
        ('Miles Davis', 'Kind of Blue'),
        ('Miles Davis', 'Sketches of Spain'),
        ('Art Blakey & The Jazz Messengers', 'Mosaic'),
        ('Charles Mingus', "Mingus Ah Um"),  # lots of songs, should have some.
        ('The Dave Brubeck Quartet', "Time Out"),
        ('Ornette Coleman', 'The Shape of Jazz to Come')  # very unusual music--should probably be easy.
    ]
recording_traits = fetch_recording_traits()


In [ ]:
artist, album = album_experiments[5]

def get_originals(artist, album):
    df = recording_traits.query(f"artist == '{artist}'").query(f"album == '{album}'")
    expected_date = df['release_date'].min()
    return df[df['release_date'] == expected_date]

get_originals(*album_experiments[1])

In [ ]:


def match_recording_traits(recording_traits, artist=None, album=None):
    def starts_with_lower(key, match):
        value = recording_traits[key]
        return value.str.contains(match, flags=re.IGNORECASE)

    if artist and album:
        return recording_traits[starts_with_lower('artist', artist) & starts_with_lower('album', album)]
    elif artist:
        return recording_traits[starts_with_lower('artist', artist)]
    elif album:
        return recording_traits[starts_with_lower('album', album)]


In [ ]:
import psycopg
query = """
            SELECT DISTINCT  -- there are duplicates in recording_to_performer where a musican plays two instuments (Louis)
                recording_to_performer.artist_id,
                recording_to_performer.instrument,
                jazz_recordings.recording_id
            FROM
                jazz_recordings
            JOIN
                recording_to_performer ON jazz_recordings.recording_id = recording_to_performer.recording_id
            -- WHERE jazz_recordings.release_date >= %(start)s
            --     AND jazz_recordings.release_date < %(end)s
        """
query = """
            SELECT
                recording_to_performer.*,
                jazz_recordings.recording_id
            FROM
                jazz_recordings
            JOIN
                recording_to_performer ON jazz_recordings.recording_id = recording_to_performer.recording_id
            -- WHERE jazz_recordings.release_date >= %(start)s
            --     AND jazz_recordings.release_date < %(end)s
            -- LIMIT 10
        """
# with psycopg.connect("dbname=musicbrainz_db user=philosofool") as conn:
#     query_result = pd.read_sql(query, conn)
# query_result

In [ ]:
autumn_leaves = """
SELECT
    jazz_recordings.*
    -- recording_to_performer.*
FROM
    jazz_recordings
-- LEFT JOIN
   -- recording_to_performer ON jazz_recordings.recording_id = recording_to_performer.recording_id
WHERE
    -- jazz_recordings.recording_id IN (4920193, 7100264, 5950710, 6925518, 29157716)
    jazz_recordings.release_group_id in (437058, 526564, 614232, 630197, 2620257)
"""
with psycopg.connect("dbname=musicbrainz_db user=philosofool") as conn:
    query_result = pd.read_sql(autumn_leaves, conn)
query_result#.duplicated(subset=['artist_name'])

In [ ]:
query_result[query_result.artist_name.str.startswith('Ella')]

In [ ]:
for row in query_result.instrument.value_counts():
    ...
for key, value in query_result.instrument.value_counts().items():
    print(key, value)

In [ ]:
query_result.album.loc[0]

In [ ]:
for row in query_result.itertuples():
    print(discogs.matching_discog(row))

In [ ]:
from jazz_graph.etl.extract_discogs import MatchDiscogs, InMemDiscogs, is_jazz_album
discogs = MatchDiscogs(InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album))

In [ ]:
discogs.matching_discog((None, None, "Adam's Apple", "Adam's Apple", "Wayne Shorter"))

In [ ]:
embeddings = torch.tensor([
    [.1, .1],
    [1, 0],
    [.1, .2],
    [.2, .1],
    [.3, .3]
])

dp = embeddings @ embeddings[[1, 3]].T
dp
dp.sum(dim=-1)

In [ ]:
from jazz_graph.data.graph_builder.make_jazz import INSTRUMENT_CATEGORIES, INDEXED_INSTRUMENTS, PerformanceFeatures

pd.Series(INSTRUMENT_CATEGORIES).value_counts()

In [ ]:
from torch_geometric.nn import GATv2Conv, GATConv, SAGEConv

x_dict = torch.tensor([
    [.1, .2, .3, .4],
    [.1, -.3, .5, .6]
])
edge_index_dict = torch.tensor([
    [1, 0], [0, 1]
])
edge_attrs = torch.tensor([
    [-1., .1, -.8, .2],
    [-1., .1, .8, .2]
])

conv = GATv2Conv(4, 6, edge_dim=4)
conv(x_dict, edge_index_dict, edge_attrs)

In [ ]:
from torch import nn
from collections.abc import Sequence
from torch_geometric.nn import HeteroConv
from torch_geometric.data import HeteroData


In [ ]:
torch.tensor([]).dim()

In [ ]:
def create_spotify_datasets(directory) -> Iterable:
    dir_path = Path(directory)
    for path in dir_path.iterdir():
        if not path.is_dir():
            continue
        datadir = next(path.iterdir())
        print(datadir)

        def file_name_is_history(name: str):
            x = name.startswith('StreamingHistory_musi')
            y = name.startswith('Streaming_History_Audio')
            return x or y

        histories = [file for file in datadir.iterdir() if file_name_is_history(file.name)]
        print(histories)
        spotify_data = []
        for history in histories:
            with open(history, 'r') as f:
                data = json.load(f)
            print("adding some from", history.name)
            print(data[0])
            spotify_data.append(data)
        yield path.name, spotify_data


for _ in create_spotify_datasets(path):
    ...

In [ ]:
from jazz_graph.data.graph_builder.make_jazz import JazzDataStore, make_jazz_graph_with_style_and_edges

store = JazzDataStore('/workspace/local_data/graph_parquet_proto')
data = make_jazz_graph_with_style_and_edges(store)

JazzModelWithStylesAndEdges(
    data['performance'].num_nodes,
    data['artist'].num_nodes,
    data['song'].num_nodes,
    edge_dim={},
    hidden_dim=16,
    embed_dim=16,
    dropout=0.,
    metadata=data.metadata(),
    model_type='gatv2'
)

In [ ]:
store.load('performance_artist_edges.parquet')

In [ ]:



from pydantic import Json



test_case = {
    'nested': {'key': [
        {('tuple_key',): 1234},
        None,
        {1: 'one'}
    ]},
    'num_performances': 154,
    'num_artists': 67,
    'num_songs': 82,
    'hidden_dim': 128,
    'embed_dim': 64,
    'edge_dim': {('artist', 'performs', 'performance'): 64},
    'projection_dim': 64,
    'style_projection_dim': 64,
    'dropout': 0.0,
    'model_type': 'gatv2',
    'num_layers': 3,
}

assert read_serialized(make_serializable(test_case)) == test_case

In [ ]:
from torch_geometric.utils import remove_isolated_nodes

nodes = torch.tensor([.1, .2, .3, .4])
edge_index = torch.tensor([[0, 1, 3],
                            [1, 0, 0]])
edge_index, edge_attr, mask = remove_isolated_nodes(edge_index, num_nodes=nodes.size(0))
mask # node mask (2 nodes)
